# Vanilla LSTM -- Feature Engineering Evaluation

**Minimal pipeline**: only a `VanillaLSTM` model is trained.
We use **F1 score** (positive class) as the headline metric because
the task is imbalanced and we care about recovering "helpful" reviews.

**Goal of this notebook:**

1. Train a single `VanillaLSTM` on cleaned review text.
2. Ablate each **text feature-engineering step** (HTML stripping,
   lowercasing, punctuation removal, stopword removal, lemmatization,
   combined title, min-word-length filter) by toggling it OFF and
   retraining.
3. Compare F1 scores -- if F1 drops, the FE step helps; if unchanged,
   it is noise; if F1 rises, it is harmful.


In [ ]:
import sys, subprocess, warnings, re, html, time
warnings.filterwarnings("ignore")

packages = ["numpy", "pandas", "scikit-learn", "torch", "nltk", "matplotlib"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

import nltk
for res in ["stopwords", "wordnet"]:
    nltk.download(res, quiet=True)

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()


## 1. Load Data

In [ ]:
cwd = Path.cwd()
workspace = cwd if (cwd / "data").exists() else cwd.parent
data_path = workspace / "data" / "processed" / "amazon_reviews_s10.parquet"
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}")

df = pd.read_parquet(data_path)

df["review_text"] = df.get("review_text", "").fillna("").astype(str)
df["title"]       = df.get("title", "").fillna("").astype(str)

print("Raw shape:", df.shape)
print("Available columns:", list(df.columns)[:15], "...")
df[["review_text", "title"]].head(3)


## 2. Create Binary Target (`helpful`)

In [ ]:
threshold = 1
candidate_cols = ["helpful_vote", "helpful_votes", "is_helpful", "helpful", "vote", "votes"]
vote_col = next((c for c in candidate_cols if c in df.columns), None)
if vote_col is None:
    raise KeyError(f"No helpful-vote column found. Tried: {candidate_cols}")

helpful_raw = pd.to_numeric(df[vote_col], errors="coerce").fillna(0)
uniq = set(np.unique(helpful_raw.astype(int)))
if uniq.issubset({0, 1}):
    df["helpful"] = helpful_raw.astype(int)
    print(f"Using `{vote_col}` directly as binary label.")
else:
    df["helpful"] = (helpful_raw > threshold).astype(int)
    print(f"Derived `helpful` = ({vote_col} > {threshold}).")

print("\nLabel distribution:")
print(df["helpful"].value_counts())
print(df["helpful"].value_counts(normalize=True).round(3).to_string())


## 3. Feature Engineering (Text Preprocessing)

This is the *only* feature engineering a text-only LSTM sees.
Each step is controlled by a flag in `DEFAULT_FE`, so we can ablate
individual steps later without touching the preprocessing code.

| Flag | Does |
|------|------|
| `strip_html` | Remove HTML tags + decode entities |
| `strip_urls` | Remove `http://...` / `www....` links |
| `lowercase` | Lowercase everything |
| `remove_punct` | Keep only letters + `[SEP]` token |
| `remove_stopwords` | Drop English stopwords (`the`, `and`, ...) |
| `lemmatize` | Map words to their dictionary form |
| `min_len` | Drop tokens with `len <= min_len` |
| `use_title` | Concatenate `title` to the review via `[SEP]` |


In [ ]:
HTML_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
PUNCT_RE = re.compile(r"[^a-zA-Z\s\[\]]")

DEFAULT_FE = dict(
    strip_html=True,
    strip_urls=True,
    lowercase=True,
    remove_punct=True,
    remove_stopwords=True,
    lemmatize=True,
    min_len=1,
    use_title=True,
)

def build_raw_text(row, use_title):
    if use_title and row["title"]:
        return f"{row['review_text']} [SEP] {row['title']}"
    return row["review_text"]

def clean_with_config(text: str, cfg: dict) -> str:
    text = html.unescape(str(text))
    if cfg["strip_html"]:
        text = HTML_RE.sub(" ", text)
    if cfg["strip_urls"]:
        text = URL_RE.sub(" ", text)
    if cfg["lowercase"]:
        text = text.lower()
    if cfg["remove_punct"]:
        text = PUNCT_RE.sub(" ", text)
    text = text.replace("[ sep ]", " [SEP] ").replace("[ SEP ]", " [SEP] ")

    out = []
    for w in text.split():
        if w.upper() == "[SEP]":
            out.append("[SEP]"); continue
        if cfg["remove_stopwords"] and w.lower() in STOP_WORDS:
            continue
        if len(w) <= cfg["min_len"]:
            continue
        if cfg["lemmatize"]:
            w = LEMMATIZER.lemmatize(w.lower() if cfg["lowercase"] else w)
        out.append(w)
    return " ".join(out) or "empty"


def apply_feature_engineering(df, cfg):
    raw = df.apply(lambda r: build_raw_text(r, cfg["use_title"]), axis=1)
    cleaned = raw.apply(lambda t: clean_with_config(t, cfg))
    X = np.asarray(cleaned.tolist(), dtype=object)
    y = np.asarray(df["helpful"].tolist(), dtype=np.int64)
    return X, y


print("=" * 60)
print("Example cleaning with DEFAULT config:")
print("=" * 60)
for i in range(2):
    raw = build_raw_text(df.iloc[i], DEFAULT_FE["use_title"])[:180]
    cleaned = clean_with_config(raw, DEFAULT_FE)[:180]
    print(f"\nRaw    : {raw}")
    print(f"Cleaned: {cleaned}")


## 4. Reusable Pipeline: FE config -> trained model -> F1 on hold-out

In [ ]:
MAX_VOCAB = 50_000
MAX_LEN = 200
PAD_ID = 0
OOV_ID = 1
EMB_DIM = 128
HID_DIM = 128
BATCH_SIZE = 256
EPOCHS = 3
PATIENCE = 2


class VanillaLSTM(nn.Module):
    '''Unidirectional LSTM with random embeddings -- text only.'''
    def __init__(self, vocab_sz, emb_dim=EMB_DIM, hid_dim=HID_DIM, pad_idx=PAD_ID, drop=0.3):
        super().__init__()
        self.emb  = nn.Embedding(vocab_sz, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.drop = nn.Dropout(drop)
        self.fc   = nn.Linear(hid_dim, 1)

    def forward(self, text):
        _, (h, _) = self.lstm(self.emb(text))
        return self.fc(self.drop(h[-1])).squeeze(1)


def tokenize_and_pad(X_train_text, X_test_text):
    counter = Counter()
    for t in X_train_text:
        counter.update(t.split())
    most_common = counter.most_common(MAX_VOCAB - 2)
    word2idx = {w: i + 2 for i, (w, _) in enumerate(most_common)}
    vocab_size = len(word2idx) + 2

    def to_ids(t):
        ids = [word2idx.get(tok, OOV_ID) for tok in t.split()][:MAX_LEN]
        ids += [PAD_ID] * (MAX_LEN - len(ids))
        return ids

    X_tr_pad = np.array([to_ids(t) for t in X_train_text], dtype=np.int64)
    X_te_pad = np.array([to_ids(t) for t in X_test_text],  dtype=np.int64)
    return X_tr_pad, X_te_pad, vocab_size


def make_text_loader(X_pad, y, batch_size, shuffle):
    ds = TensorDataset(
        torch.tensor(X_pad, dtype=torch.long),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_vanilla(model, train_loader, val_loader, pw, epochs, patience, name):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=1,
    )
    best_val, wait, best_state = float("inf"), 0, None
    for ep in range(1, epochs + 1):
        model.train()
        s_loss, s_n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            s_loss += loss.item() * xb.size(0); s_n += xb.size(0)

        model.eval()
        v_loss, v_n = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb), yb)
                v_loss += loss.item() * xb.size(0); v_n += xb.size(0)
        tl, vl = s_loss / s_n, v_loss / v_n
        scheduler.step(vl)
        print(f"  [{name}] Epoch {ep}/{epochs}  loss={tl:.4f}  val_loss={vl:.4f}")
        if vl < best_val:
            best_val, wait = vl, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {ep}"); break
    if best_state:
        model.load_state_dict(best_state)


@torch.no_grad()
def evaluate_f1(model, loader):
    model.eval()
    probs, trues = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        probs.extend(torch.sigmoid(model(xb)).cpu().numpy().tolist())
        trues.extend(yb.numpy().tolist())
    yp = np.array(probs)
    yt = np.array(trues, dtype=int)
    ypred = (yp >= 0.5).astype(int)
    return dict(
        accuracy  = accuracy_score(yt, ypred),
        precision = precision_score(yt, ypred, zero_division=0),
        recall    = recall_score(yt, ypred, zero_division=0),
        f1        = f1_score(yt, ypred, zero_division=0),
        y_true=yt, y_pred=ypred, y_prob=yp,
    )


def run_pipeline(cfg, tag, epochs=EPOCHS, patience=PATIENCE):
    t0 = time.time()
    print(f"\n>>> Running pipeline: {tag}")
    print(f"    FE config: {cfg}")

    X_text, y = apply_feature_engineering(df, cfg)
    X_tr_txt, X_te_txt, y_tr_all, y_te = train_test_split(
        X_text, y, test_size=0.2, shuffle=True, random_state=SEED, stratify=y,
    )
    X_tr_pad, X_te_pad, vocab_size = tokenize_and_pad(X_tr_txt, X_te_txt)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr_pad, y_tr_all, test_size=0.15, shuffle=True,
        random_state=SEED, stratify=y_tr_all,
    )
    n_neg, n_pos = (y_tr == 0).sum(), (y_tr == 1).sum()
    pw = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

    train_loader = make_text_loader(X_tr,      y_tr,  BATCH_SIZE, True)
    val_loader   = make_text_loader(X_val,     y_val, BATCH_SIZE, False)
    test_loader  = make_text_loader(X_te_pad,  y_te,  512,        False)

    torch.manual_seed(SEED); np.random.seed(SEED)
    model = VanillaLSTM(vocab_sz=vocab_size).to(device)
    train_vanilla(model, train_loader, val_loader, pw, epochs, patience, tag)
    metrics = evaluate_f1(model, test_loader)
    metrics["vocab_size"] = int(vocab_size)
    metrics["minutes"]    = (time.time() - t0) / 60

    print(f"    -> F1={metrics['f1']:.4f}  Acc={metrics['accuracy']:.4f}  "
          f"P={metrics['precision']:.4f}  R={metrics['recall']:.4f}  "
          f"({metrics['minutes']:.1f} min)")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics


## 5. Train the Full Vanilla LSTM (all FE steps ON)

This is the **baseline** we compare every ablation against.


In [ ]:
full_metrics = run_pipeline(DEFAULT_FE, tag="Full FE")

print("\n" + "=" * 60)
print("VANILLA LSTM  --  Full Feature Engineering")
print("=" * 60)
print(f"  Accuracy  : {full_metrics['accuracy']:.4f}")
print(f"  Precision : {full_metrics['precision']:.4f}")
print(f"  Recall    : {full_metrics['recall']:.4f}")
print(f"  F1-score  : {full_metrics['f1']:.4f}")

print("\nClassification report:")
print(classification_report(
    full_metrics["y_true"], full_metrics["y_pred"],
    target_names=["Not Helpful", "Helpful"], digits=4,
))

cm = confusion_matrix(full_metrics["y_true"], full_metrics["y_pred"])
print("Confusion matrix:\n", cm)


## 6. Feature-Engineering Ablation

For each FE flag we toggle it **off** (or set `min_len = 0` / `use_title = False`),
keep everything else at the default, and retrain the Vanilla LSTM from scratch
with the same seed and epoch budget.

**Reading the table:**

- F1 **drops** a lot -> FE step is **important** (keep it)
- F1 **unchanged** -> FE step is **noise** (safe to drop)
- F1 **rises** -> FE step is **harmful** (remove it for a free win)


In [ ]:
ABLATIONS = [
    ("No HTML strip",       {"strip_html": False}),
    ("No URL strip",        {"strip_urls": False}),
    ("No lowercasing",      {"lowercase": False}),
    ("Keep punctuation",    {"remove_punct": False}),
    ("Keep stopwords",      {"remove_stopwords": False}),
    ("No lemmatization",    {"lemmatize": False}),
    ("min_len=0 (keep 1-char tokens)", {"min_len": 0}),
    ("No title concat",     {"use_title": False}),
]

ablation_results = {"Full FE (baseline)": full_metrics}

for name, overrides in ABLATIONS:
    cfg = {**DEFAULT_FE, **overrides}
    ablation_results[name] = run_pipeline(cfg, tag=name)

rows = []
base_f1 = full_metrics["f1"]
for name, m in ablation_results.items():
    delta = m["f1"] - base_f1
    if name == "Full FE (baseline)":
        verdict = "baseline"
    elif delta <= -0.005:
        verdict = "IMPORTANT (F1 drops)"
    elif delta >= 0.005:
        verdict = "HARMFUL (F1 rises)"
    else:
        verdict = "noise (no effect)"
    rows.append(dict(
        Run=name,
        F1=round(m["f1"], 4),
        Acc=round(m["accuracy"], 4),
        Precision=round(m["precision"], 4),
        Recall=round(m["recall"], 4),
        dF1=round(delta, 4),
        Verdict=verdict,
    ))

ablation_df = pd.DataFrame(rows).sort_values("dF1")
print("\n" + "=" * 78)
print("FEATURE-ENGINEERING ABLATION  (sorted; most-negative dF1 = most important)")
print("=" * 78)
print(ablation_df.to_string(index=False))


## 7. Visualise dF1 per Feature-Engineering Step

In [ ]:
plot_df = ablation_df[ablation_df["Run"] != "Full FE (baseline)"].copy()
colors = [
    "#d62728" if d <= -0.005 else ("#2ca02c" if d >= 0.005 else "#7f7f7f")
    for d in plot_df["dF1"]
]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(plot_df["Run"], plot_df["dF1"], color=colors, edgecolor="black")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("delta F1 vs Full FE baseline")
ax.set_title(f"Feature-Engineering Ablation  (Vanilla LSTM, baseline F1={base_f1:.4f})")
for y, d in enumerate(plot_df["dF1"]):
    ax.text(d, y, f"  {d:+.4f}", va="center",
            ha="left" if d >= 0 else "right", fontsize=10)
plt.tight_layout()
plt.show()

print("\nLegend: red = important FE step (removing it hurts F1),"
      " grey = noise, green = harmful (removing it helps).")
